# 04 · 衍生:从 SFT 走向强化学习

> 这是本仓库的**收尾与衔接**章节,只讲概念、不做实现。
> 强化学习的动手代码会放到**下一个仓库**里,本章帮你建立完整的心智地图。

前面 `01`~`03` 我们做的都是 **SFT(Supervised Fine-Tuning,监督微调)**:给模型看「问题 → 标准答案」,让它模仿。LoRA/QLoRA 只是 SFT 的高效实现方式。

但 SFT 有天花板 —— 这正是强化学习登场的原因。

## 一、SFT 的局限

SFT 的本质是**模仿**给定的答案,它有几个学不到的东西:

1. **只会模仿,不懂「好坏排序」。** 训练数据里每个问题通常只有一个「标准答案」,模型学会照抄,但学不到「答案 A 比答案 B 好」这种相对偏好。
2. **难以对齐人类的细腻偏好。** 比如「更礼貌」「更简洁」「更安全、不要有害」,这些很难用一堆标准答案穷举,却很容易由人「二选一」地判断。
3. **暴露偏差(exposure bias)。** SFT 永远在「正确前缀」上训练,而真实生成时模型要面对自己产生的、可能有瑕疵的前缀,SFT 没针对这种情况优化过。

**核心矛盾:** 收集「标准答案」很贵,但让人「在两个回答里挑更好的一个」很便宜。
强化学习(尤其是 RLHF)正是把这种**廉价的偏好信号**变成训练信号的框架。

## 二、RLHF 全景图

主流大模型的对齐通常分三步:**SFT → 训练奖励模型 → 用强化学习优化策略**。

```mermaid
flowchart TD
    pretrain["预训练底座<br/>(会说话,但不会聊天)"] --> sft["第一步:SFT<br/>本仓库做的<br/>LoRA/QLoRA 指令微调"]
    sft --> collect["收集人类偏好<br/>对同一问题的多个回答排序"]
    collect --> rm["第二步:训练奖励模型 RM<br/>学会给回答打分"]
    sft --> policy["策略模型<br/>(初始 = SFT 模型)"]
    rm --> rl["第三步:强化学习<br/>PPO / GRPO 等"]
    policy --> rl
    rl --> aligned["对齐后的模型<br/>更符合人类偏好"]
```

关键角色:

- **策略模型 (Policy):** 我们要优化的语言模型,初始就是 SFT 得到的模型。
- **奖励模型 (Reward Model):** 输入一个「问题+回答」,输出一个分数,代表人类有多喜欢。
- **参考模型 (Reference):** 通常是冻结的 SFT 模型,用来约束策略「别跑太偏」(KL 惩罚)。

## 三、两条主流路线:PPO 与 DPO

### 路线 A:RLHF + PPO(经典路线)

1. 训练一个**奖励模型**给回答打分。
2. 用 **PPO(Proximal Policy Optimization)** 这种强化学习算法优化策略模型:
   - 策略模型生成回答 → 奖励模型打分 → 用分数(reward)更新策略,让高分回答的概率变大;
   - 同时加一个 **KL 惩罚**,防止策略偏离原 SFT 模型太远(避免「为了骗高分」说胡话)。

优点:效果强、上限高。缺点:要同时维护 4 个模型(策略/参考/奖励/价值),流程复杂、训练不稳、显存吃紧。

### 路线 B:DPO(更轻量,近年流行)

**DPO(Direct Preference Optimization)** 的洞见:**可以跳过「显式训练奖励模型 + 跑 PPO」,直接用偏好数据优化策略。**

- 数据形式:每条是 `(问题, 更好的回答 chosen, 更差的回答 rejected)`。
- 用一个巧妙的损失函数,直接拉高 `chosen` 相对 `rejected` 的概率(相对参考模型)。
- 不需要单独的奖励模型,也不用采样式的 RL,**更稳定、更省资源**,和 LoRA/QLoRA 结合得很好。

一句话对比:**PPO 是「在线试错 + 打分」,DPO 是「直接从成对偏好里学」。**

## 四、抢先看:数据长什么样

只是让你对「偏好数据」和「TRL 训练器」有个直观印象,**下面的代码只作示意,本仓库不运行**。

SFT 数据(本仓库用的,一问一答):

```json
{"instruction": "什么是过拟合?", "output": "过拟合是指模型..."}
```

偏好数据(DPO 用的,一问 + 一好一坏两个答案):

```json
{
  "prompt": "什么是过拟合?",
  "chosen": "过拟合是指模型把训练数据的噪声也学了进去,导致泛化变差。",
  "rejected": "过拟合就是训练太多次了。"
}
```

用 TRL 做 DPO 的骨架(示意):

```python
# 下一个仓库会真正跑通这段
from trl import DPOConfig, DPOTrainer

trainer = DPOTrainer(
    model=policy_model,            # 初始为本仓库 SFT 出来的模型(可带 LoRA)
    ref_model=None,                # 用 LoRA 时可省略,内部用禁用适配器当参考
    args=DPOConfig(beta=0.1, ...), # beta 控制偏离参考模型的强度
    train_dataset=preference_ds,   # 上面的 chosen/rejected 数据
    processing_class=tokenizer,
)
trainer.train()
```

可以看到:**DPO 完全可以接在本仓库的 QLoRA 之上** —— 底座 4-bit、只训练 LoRA 适配器,单张 16GB 卡也能做偏好对齐。

## 五、本仓库到此为止 —— 下一个仓库预告

**本仓库(LoRA / QLoRA / PEFT)已完结。** 你已经掌握:

- PEFT 思想与 LoRA 低秩分解原理
- QLoRA 的 4-bit / NF4 量化,以及在 16GB 上微调 7B
- 适配器的加载、前后对比、合并部署
- 以及「为什么 SFT 之后还需要强化学习」

**下一个仓库将动手实现强化学习对齐,预计包含:**

1. **奖励模型 (Reward Model)**:用偏好数据训练一个打分模型。
2. **DPO 实战**:用 `trl` 的 `DPOTrainer` + LoRA/QLoRA,做一次完整的偏好对齐(推荐作为入门首选,稳定省资源)。
3. **PPO 实战**:用 `PPOTrainer` 走一遍经典 RLHF 流程,理解 reward / KL 惩罚 / 价值网络。
4. （可选)**GRPO** 等更新的、面向推理任务的强化学习方法。

到时会延续这里的习惯:**中文讲解 + Blackwell 环境 + 尽量在单张 16GB 卡上跑通。**

> 承上启下:本仓库训练出的 SFT 模型,正是下一个仓库里强化学习的**起点(策略模型的初始化)**。